In [1]:
!pip install qiskit qiskit-ibm-runtime qiskit-aer numpy

   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   --------- ------------------------------ 2.1/9.3 MB 11.8 MB/s eta 0:00:01
   ----------------------- ---------------- 5.5/9.3 MB 14.0 MB/s eta 0:00:01
   -------------------------------------- - 8.9/9.3 MB 14.6 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 12.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 13.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.9 MB 16.8 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.9 MB 16.1 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 16.3 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 15.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ----------------------------


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [58]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import PermutationGate
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
import numpy as np

In [59]:
def match_operator(n: int, char_size: int) -> QuantumCircuit:
    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")
    qout = QuantumRegister(n, "o")
    qc = QuantumCircuit(qx, qy, qout)

    for i in range(n * char_size):
        qc.cx(qx[i], qy[i])
        qc.x(qy[i])

    for i in range(0, n):
        ctrls = list(range(i * char_size, i * char_size + char_size))
        qc.mcx([*qy[ctrls]], qout[i])
    return qc


In [60]:
def extension_operator(n: int, i: int) -> QuantumCircuit:
    """
    builds the quantum circuit for the extension operator for an array of size `n`
    for `i`-th lambda/L vector
    """

    shift = 2 ** (i - 1)

    qin = QuantumRegister(n, "in")
    qout = QuantumRegister(n, "out")
    qc = QuantumCircuit(qin, qout)

    for j in range(n - shift):
        qc.ccx(qin[j], qin[j + shift], qout[j])

    return qc


def quantum_rotation(n: int, s: int):
    """
    builds the rotation operator on `n` qubits by `s` positions
    """
    s %= n
    qc = QuantumCircuit(n)

    perm = [((k - s) % n) for k in range(n)]

    qc.append(PermutationGate(perm), range(n))
    return qc.decompose().to_gate(label=f"R_{s}")


def quantum_parametric_rotation(n: int) -> QuantumCircuit:
    """
    builds the quantum circuit for the parametric rotation gate on `n` qubits.
    The amount of control qubits it's equal to ceil(log2(n))
    """

    l = int(np.ceil(np.log2(n)))

    jr = QuantumRegister(l, "j")
    qr = QuantumRegister(n, "q")
    qc = QuantumCircuit(jr, qr)

    for i in range(l):
        qc = qc.compose(quantum_rotation(n, 2**i).control(1), [jr[i], *qr])
    return qc


def quantum_character_rotation(n: int, char_size: int) -> QuantumCircuit:
    """
    builds the quantum circuit for the parametric rotation on characters of
    size `char_size` qubits of `n` positions
    """
    
    l = int(np.ceil(np.log2(n)))

    jr = QuantumRegister(l, "j")
    qr = QuantumRegister(n * char_size, "q")
    qc = QuantumCircuit(jr, qr)

    for i in range(l):
        controlled_rot = quantum_rotation(n, 2**i).control(1)

        for bit in range(char_size):
            lines = [qr[k * char_size + bit] for k in range(n)]
            qc = qc.compose(controlled_rot, [jr[i], *lines])

    return qc


def copy_operator(n: int):
    """
    builds a copy gate on `n` qubits
    """
    
    qin = QuantumRegister(n, "in")
    qout = QuantumRegister(n, "out")
    qc = QuantumCircuit(qin, qout)

    for i in range(n):
        qc.cx(qin[i], qout[i])

    return qc.to_gate(label="COPY")


def or_operator(n: int):
    """
    builds a logical or gate on `n` qubits
    """
    
    qx = QuantumRegister(n, "x")
    qy = QuantumRegister(1, "y")
    qc = QuantumCircuit(qx, qy)

    qc.x(qx)
    qc.mcx(qx, qy)
    qc.x(qx)
    qc.x(qy)

    return qc.to_gate(label="OR")


def contr_bitwise_and_operator(n: int):
    """
    builds a controlled and operator on `n` qubits
    """
    
    qc_ctrl = QuantumRegister(1, "c")
    qL = QuantumRegister(n, "L")
    qin = QuantumRegister(n + 1, "qin")
    qout = QuantumRegister(n + 1, "qout")
    qc = QuantumCircuit(qc_ctrl, qL, qin, qout)

    for k in range(n):
        qc.mcx([qc_ctrl[0], qL[k], qin[k]], qout[k])

    return qc.to_gate(label="C-AND")

In [61]:
def FSM(n, char_size) -> QuantumCircuit:
    """
    builds the Fixed Substring Matching circuit for inputs of size `n` and
    characters of `char_size` qubits
    """

    d_len = int(np.ceil(np.log2(n)))
    L_num = d_len
    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")
    qd = QuantumRegister(d_len, "d")
    qD = QuantumRegister(d_len + 1, "D")
    qL = []

    for i in range(L_num):
        qL.append(QuantumRegister(n, "L[" + str(i) + "]"))

    qD = []
    for i in range(d_len + 1):
        qD.append(QuantumRegister(n + 1, "D[" + str(i - 1) + "]"))

    qr = QuantumRegister(1, "r")
    qc = QuantumCircuit(qx, qy, qd, *qL, *qD, qr)

    qc = qc.compose(
        match_operator(n, char_size).to_gate(label="MATCH"), [*qx, *qy, *qL[0]]
    )

    for i in range(L_num - 1):
        qc = qc.compose(
            extension_operator(n, i + 1).to_gate(label=f"EXT_{i}"),
            [*qL[i][:n], *qL[i + 1]],
        )

    for i in range(d_len):

        qc = qc.compose(
            contr_bitwise_and_operator(n), [qd[i], *qL[i], *qD[i], *qD[i + 1]]
        )

        controlled_rot = quantum_rotation(n + 1, 2**i).control(1)
        qc = qc.compose(controlled_rot, [qd[i], *qD[i + 1]])
        qc.x(qd[i])
        controlled_copy = copy_operator(n + 1).control(1)
        qc = qc.compose(controlled_copy, [qd[i], *qD[i], *qD[i + 1]])
        qc.x(qd[i])

    qc = qc.compose(or_operator(n + 1), [*qD[d_len], qr])

    return qc


In [62]:
def SFC(n, char_size) -> QuantumCircuit:
    """
    builds the Shared Fix Substring Check circuit for inputs of size `n` and
    characters of `char_size` qubits
    """

    d_len = int(np.ceil(np.log2(n)))
    L_num = d_len

    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")
    qd = QuantumRegister(d_len, "d")
    qD = QuantumRegister(d_len + 1, "D")
    qL = []

    for i in range(L_num):
        qL.append(QuantumRegister(n, "L[" + str(i) + "]"))

    qD = []
    for i in range(d_len + 1):
        qD.append(QuantumRegister(n + 1, "D[" + str(i - 1) + "]"))

    qr = QuantumRegister(1, "r")
    qc = QuantumCircuit(qx, qy, qd, *qL, *qD, qr)

    qc.x(qD[0])

    qc = qc.compose(FSM(n, char_size))
    return qc


In [63]:
def FPM(n, char_size) -> QuantumCircuit:
    """
    builds the Fixed Prefix Matching operator for inputs of size `n`
    """
    
    d_len = int(np.ceil(np.log2(n)))
    L_num = d_len
    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")
    qd = QuantumRegister(d_len, "d")
    qD = QuantumRegister(d_len + 1, "D")
    qL = []

    for i in range(L_num):
        qL.append(QuantumRegister(n, "L[" + str(i) + "]"))

    qD = []
    for i in range(d_len + 1):
        qD.append(QuantumRegister(n + 1, "D[" + str(i - 1) + "]"))

    qr = QuantumRegister(1, "r")
    qc = QuantumCircuit(qx, qy, qd, *qL, *qD, qr)

    # init
    qc.x(qD[0][0])

    qc = qc.compose(FSM(n, char_size))
    return qc


In [64]:
def grover_operator(n: int):
    """
    builds the diffusion operator from the grover algorithm
    """

    qx = QuantumRegister(n, "x")
    qc = QuantumCircuit(qx)

    qc.h(qx)
    qc.x(qx)

    if n == 1:
        qc.z(qx[0])
    else:
        qc.h(qx[n - 1])
        qc.mcx(qx[list(range(n - 1))], qx[n - 1])
        qc.h(qx[n - 1])

    qc.x(qx)
    qc.h(qx)

    return qc.to_gate(label="DIFFUSION")


def int_to_bin(num: int, code_len: int) -> str:
    """
    converts `num` in a binary string, with the specified `code_len`
    """

    num = bin(num)[2:]
    num = "0" * (code_len - len(num)) + num
    return num


def quantum_number_encode(num: int, code_len: int = 0):
    """
    encodes the given `num` into a gate. if specified, `code_len` lets
    you use more qubits than needed for the number specified by `num`
    """
    
    if num == 0:
        return QuantumCircuit(code_len).to_gate(label=" 0 ")

    if code_len < int(np.ceil(np.log2(num))):
        code_len = len(bin(num)[2:])
    
    bin_num = int_to_bin(num, code_len)
    qc = QuantumCircuit(code_len)

    for c, i in enumerate(bin_num):
        if i == "1":
            qc.x(code_len - 1 - c)

    return qc.to_gate(label=" " + str(num) + " ")


def encode_boolean_string(x: str) -> QuantumCircuit:
    """
    encodes the given boolean string `x` in a quantum circuit
    """
    n = len(x)
    qx = QuantumRegister(n, "x")
    qc = QuantumCircuit(qx)

    for c, i in enumerate(x):
        if i == "1":
            qc.x(qx[c])

    return qc


In [65]:
def quantum_step_circuit(x: str, y: str, n: int, d: int, char_size: int, grover_epochs: int) -> QuantumCircuit:
    """
    builds the quantum circuit for the QLCS_test on the strings `x` and `y` of length `n`, 
    checking for the common substring of length `d`, with the specified `char_size`
    """
    
    qi_amount = int(np.ceil(np.log2(n)))
    
    qi = QuantumRegister(qi_amount, "i")
    qj = QuantumRegister(qi_amount, "j")
    qx = QuantumRegister(n * char_size, "x")
    qy = QuantumRegister(n * char_size, "y")
    
    ancilla_amount = n * int(np.ceil(np.log2(n))) + (n + 1) * (
        int(np.ceil(np.log2(n + 1)))
    )

    qa = QuantumRegister(ancilla_amount, "a")
    qd = QuantumRegister(qi_amount, "d")

    qexp = QuantumRegister(2, 'dom_exp')
    qr = QuantumRegister(1, "r")
    qo = QuantumRegister(1, "out")
    c = ClassicalRegister(1, "c")

    qc = QuantumCircuit(qexp, qi, qj, qx, qy, qa, qd, qr, qo, c)

    qc = qc.compose(quantum_number_encode(d, qi_amount), qd)
    qc = qc.compose(encode_boolean_string(x), qx)
    qc = qc.compose(encode_boolean_string(y), qy)

    qc.h(qexp[0])
    qc.barrier(label="init")

    # search phase
    qc.h(qj)
    qc.x(qo)
    qc.h(qo)

    for _ in range(grover_epochs):
        qc = qc.compose(
            quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qj, *qx]
        )
        qc = qc.compose(
            SFC(n, char_size).to_gate(label="SFC"), [*qx, *qy, *qd, *qa, qr]
        )

        qc.ccx(qexp[0], qr, qo)
        qc = qc.compose(
            SFC(n, char_size).inverse().to_gate(label="- SFC"), [*qx, *qy, *qd, *qa, qr]
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
            [*qj, *qx],
        )

        qc = qc.compose(grover_operator(qi_amount + 1), [qexp[0], *qj])

    for j in qj:
        qc.measure(j, c)

    qc.barrier(label="SEARCH PHASE")
    qc.h(qexp[1])
    qc.h(qi)

    # verification phase
    for _ in range(grover_epochs):
        qc = qc.compose(
            quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qj, *qx]
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qi, *qx]
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qi, *qy]
        )
        qc = qc.compose(
            FPM(n, char_size).to_gate(label="FPM"), [*qx, *qy, *qd, *qa, qr]
        )

        qc.ccx(qexp[1], qr, qo)

        qc = qc.compose(
            FPM(n, char_size).inverse().to_gate(label="- FPM"), [*qx, *qy, *qd, *qa, qr]
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
            [*qi, *qy],
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
            [*qi, *qx],
        )
        qc = qc.compose(
            quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
            [*qj, *qx],
        )

        qc = qc.compose(grover_operator(qi_amount + 1), [qexp[1], *qi])

    for i in qi:
        qc.measure(i, c)
    qc.barrier(label="VERIFICATION PHASE")

    # final check
    qc.h(qo)
    qc.x(qo)

    qc = qc.compose(
        quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qj, *qx]
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qi, *qx]
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).to_gate(label="ROT"), [*qi, *qy]
    )
    qc = qc.compose(FPM(n, char_size).to_gate(label="FPM"), [*qx, *qy, *qd, *qa, qr])
    qc.cx(qr, qo)

    qc = qc.compose(
        FPM(n, char_size).inverse().to_gate(label="- FPM"), [*qx, *qy, *qd, *qa, qr]
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
        [*qi, *qy],
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
        [*qi, *qx],
    )
    qc = qc.compose(
        quantum_character_rotation(n, char_size).inverse().to_gate(label="- ROT"),
        [*qj, *qx],
    )

    qc.measure(qo, c)
    qc.barrier(label="FINAL CHECK")

    return qc


In [66]:
def run(qc: QuantumCircuit, shots=1024):
    """
    returns the result of the execution of the `qc` quantum circuit
    """
    sim_backend = AerSimulator(method="matrix_product_state") # the only viable method for circuits of this size with no entanglement
    sim_backend.set_max_qubits(qc.num_qubits) # as big as needed

    pm = generate_preset_pass_manager(backend=sim_backend, optimization_level=2)
    isa_qc = pm.run(qc)
    sampler = Sampler(mode=sim_backend)

    job = sampler.run([isa_qc], shots=shots)
    results = job.result()
    data = results[0].data
    counts = data.c.get_counts()

    return counts


RUNS_PER_CIRCUIT = 2

def grover_iter_search(x: str, y: str, n: int, d: int, char_size: int):
    """
    implements a pseudo-BBHT approach to the grover search
    """
    m = 1
    lam = 1.34
    while True:
        j = int(m)
        qc = quantum_step_circuit(x, y, n, d, char_size, grover_epochs=j)
        c = run(qc, RUNS_PER_CIRCUIT)
        if "1" in list(c.keys()):
            return True
        
        m = min(lam * m, np.sqrt(n))
        if m >= np.sqrt(n):
            break
        
    return False

In [67]:
def quantum_LCS_test(x: str, y: str, n: int, d: int, char_size: int) -> bool:
    """
    returns the result of the QLCS-test on string `x`, `y` of size `n` for substring of length `d`
    on characters of size `char_size`
    """
    # trivial cases
    if d == 0:
        return True
    if d == n:
        return x == y 

    return grover_iter_search(x, y, n, d, char_size)


In [68]:
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Tuple


def quantum_LCS_instrumented(
    x: str, y: str, n: int, char_size: int, verbose: bool = False
) -> Tuple[int, int]:
    """
    Esegue la ricerca esponenziale senza tentativi ripetuti (single-shot per passo).
    Restituisce (lunghezza_lcs, numero_iterazioni).
    """
    if n == 0:
        return 0, 0

    iterations = 0
    l = 0
    r = n
    bound = 1

    # Fase 1: Ricerca esponenziale
    while bound <= n:
        iterations += 1
        if verbose:
            print(f"testing for common substring of length {bound}:", end=" ")

        if quantum_LCS_test(x, y, n, bound, char_size):
            l = bound
            bound *= 2
            if verbose:
                print("found!")
        else:
            r = bound - 1
            if verbose:
                print("not found!")
            break

    # Fase 2: Ricerca binaria nell'intervallo ristretto [l, r]
    while l < r:
        d = (l + r + 1) // 2
        iterations += 1
        if verbose:
            print(f"testing for common substring of length {d}:", end=" ")

        if quantum_LCS_test(x, y, n, d, char_size):
            l = d
            if verbose:
                print("found!")
        else:
            r = d - 1
            if verbose:
                print("not found!")

    return l, iterations

In [69]:
def classical_lcs_substring_len(x: str, y: str, char_size: int = 2) -> int:
    """
    for testing purpose
    """
    xs = [x[i : i + char_size] for i in range(0, len(x), char_size)]
    ys = [y[i : i + char_size] for i in range(0, len(y), char_size)]

    n = len(xs)
    m = len(ys)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    best = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if xs[i - 1] == ys[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
                best = max(best, dp[i][j])
    return best

In [70]:
import random
from typing import Dict, List, Tuple

def generate_quantum_lcs_pair(
    string_len: int,
    alphabet_size: int,
    seed: int | None = None,
) -> Tuple[str, str]:
    """
    Genera una coppia di stringhe binarie (X, Y) valide per l'algoritmo quantum_LCS.

    Parametri:
        string_len (int): Numero totale di caratteri (deve essere una potenza di 2 >= 2).
        alphabet_size (int): Numero di bit per singolo carattere (es. 2 o 3).
        seed (int | None): Seme opzionale per la riproducibilità della generazione.

    Restituisce:
        Tuple[str, str]: Coppia di stringhe binarie (X, Y) di lunghezza
                         string_len * alphabet_size bit.
    """
    if string_len < 2 or (string_len & (string_len - 1)) != 0:
        raise ValueError("string_len deve essere una potenza di 2 maggiore o uguale a 2.")
    if alphabet_size < 2:
        raise ValueError("alphabet_size deve essere almeno pari a 2.")

    rng = random.Random(seed)

    # Costruzione dell'intero alfabeto binario di dimensione 2^alphabet_size
    full_alphabet = [
        format(i, f"0{alphabet_size}b") for i in range(2**alphabet_size)
    ]

    # Riserva dei due caratteri di chiusura disgiunti
    closing_x = "0" * alphabet_size
    closing_y = "1" * alphabet_size

    # Caratteri utilizzabili per il corpo delle stringhe (esclusi i delimitatori)
    body_alphabet = [
        char for char in full_alphabet if char not in (closing_x, closing_y)
    ]

    # Generazione del corpo di (string_len - 1) caratteri + 1 carattere di chiusura
    body_len = string_len - 1
    x_body = "".join(rng.choices(body_alphabet, k=body_len))
    y_body = "".join(rng.choices(body_alphabet, k=body_len))

    return x_body + closing_x, y_body + closing_y

In [71]:
def generate_test_suite(
    alphabet_sizes: List[int] = [2, 3],
    lengths: List[int] = [2, 4, 8],
    seed: int = 42,
) -> Dict[Tuple[int, int], Tuple[str, str]]:
    """
    Genera un dizionario di test indicizzato da (alphabet_size, string_len).
    """
    dataset: Dict[Tuple[int, int], Tuple[str, str]] = {}
    for alpha_size in alphabet_sizes:
        for length in lengths:
            pair = generate_quantum_lcs_pair(
                string_len=length,
                alphabet_size=alpha_size,
                seed=seed + alpha_size * 100 + length,
            )
            dataset[(alpha_size, length)] = pair
    return dataset

In [72]:
def run_benchmark_with_error_tracking(
    alphabet_sizes: List[int] = [2, 3],
    lengths: List[int] = [2, 4, 8, 16],
    num_trials: int = 20,
    base_seed: int = 42,
) -> Dict[Tuple[int, int], Dict[str, List[int]]]:
    """
    Esegue `num_trials` prove per ogni configurazione registrando separatamente
    i passi iterativi delle esecuzioni corrette e di quelle errate, quindi
    genera un istogramma impilato per confrontarle.
    """
    results: Dict[Tuple[int, int], Dict[str, List[int]]] = {}

    for alpha_size in alphabet_sizes:
        for length in lengths:
            correct_iters: List[int] = []
            error_iters: List[int] = []

            for trial in range(num_trials):
                seed = base_seed + alpha_size * 10000 + length * 100 + trial
                X, Y = generate_quantum_lcs_pair(
                    string_len=length,
                    alphabet_size=alpha_size,
                    seed=seed,
                )

                assert len(X) == len(Y) == length * alpha_size
                assert X[-alpha_size:] != Y[-alpha_size:]

                q_result, iter_count = quantum_LCS_instrumented(
                    X, Y, length, alpha_size, verbose=False
                )
                c_result = classical_lcs_substring_len(X, Y, alpha_size)

                if q_result == c_result:
                    correct_iters.append(iter_count)
                else:
                    error_iters.append(iter_count)

            results[(alpha_size, length)] = {
                "correct": correct_iters,
                "error": error_iters,
            }

            total_mean = (sum(correct_iters) + sum(error_iters)) / num_trials
            err_rate = (len(error_iters) / num_trials) * 100
            print(
                f"[alpha={alpha_size}, len={length:2d}] "
                f"Iterazioni medie: {total_mean:.2f} | "
                f"Corretti: {len(correct_iters)}/{num_trials} | "
                f"Errori: {len(error_iters)}/{num_trials} ({err_rate:.1f}%)"
            )

    # Costruzione del grafico a barre raggruppate e impilate
    x_positions = np.arange(len(lengths))
    bar_width = 0.36

    # Palette cromatica distinta per alfabeto e stato (corretto vs errore)
    palette = {
        2: {"correct": "#1f77b4", "error": "#ff7f0e"},  # Blu / Arancione
        3: {"correct": "#2ca02c", "error": "#d62728"},  # Verde / Rosso
    }

    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=120)
    max_bar_height = 0.0

    for idx, alpha_size in enumerate(alphabet_sizes):
        ok_contrib = [
            sum(results[(alpha_size, l)]["correct"]) / num_trials for l in lengths
        ]
        err_contrib = [
            sum(results[(alpha_size, l)]["error"]) / num_trials for l in lengths
        ]
        err_counts = [len(results[(alpha_size, l)]["error"]) for l in lengths]

        offset = (idx - (len(alphabet_sizes) - 1) / 2) * bar_width
        colors = palette.get(
            alpha_size, {"correct": "#4c72b0", "error": "#c44e52"}
        )

        # Regione inferiore: iterazioni da esecuzioni corrette
        bars_ok = ax.bar(
            x_positions + offset,
            ok_contrib,
            width=bar_width,
            color=colors["correct"],
            edgecolor="black",
            linewidth=0.8,
            label=f"Alfabeto {alpha_size} bit — Esecuzioni corrette",
        )

        # Regione superiore (impilata): iterazioni da esecuzioni errate
        bars_err = ax.bar(
            x_positions + offset,
            err_contrib,
            width=bar_width,
            bottom=ok_contrib,
            color=colors["error"],
            edgecolor="black",
            linewidth=0.8,
            hatch="//",
            label=f"Alfabeto {alpha_size} bit — Esecuzioni errate",
        )

        # Annotazioni numeriche sulle singole regioni e sul totale
        for i in range(len(lengths)):
            h_ok = ok_contrib[i]
            h_err = err_contrib[i]
            h_tot = h_ok + h_err
            max_bar_height = max(max_bar_height, h_tot)

            # Etichetta interna alla regione corretta (se sufficientemente alta)
            if h_ok >= 0.25:
                ax.text(
                    x_positions[i] + offset,
                    h_ok / 2,
                    f"{h_ok:.2f}",
                    ha="center",
                    va="center",
                    color="white",
                    fontsize=8.5,
                    fontweight="bold",
                )

            # Etichetta interna alla regione di errore (se presente)
            if h_err > 0:
                ax.text(
                    x_positions[i] + offset,
                    h_ok + h_err / 2,
                    f"{h_err:.2f}",
                    ha="center",
                    va="center",
                    color="white",
                    fontsize=8,
                    fontweight="bold",
                )

            # Etichetta complessiva sopra la barra con conteggio errori
            header_label = (
                f"{h_tot:.2f}\n({err_counts[i]} err)"
                if err_counts[i] > 0
                else f"{h_tot:.2f}"
            )
            ax.annotate(
                header_label,
                xy=(x_positions[i] + offset, h_tot),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8.5,
            )

    ax.set_title(
        f"Passi iterativi medi di quantum_LCS: contributo esecuzioni corrette vs errate ({num_trials} prove)",
        fontsize=11.5,
        pad=14,
    )
    ax.set_xlabel("Lunghezza stringa in caratteri (string_len)", fontsize=10)
    ax.set_ylabel("Passi iterativi medi per esecuzione", fontsize=10)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f"{l} ({l * 2}/{l * 3} bit)" for l in lengths])
    ax.set_ylim(0, max_bar_height + 1.1)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.legend(loc="upper left", frameon=True, fontsize=9)

    plt.tight_layout()
    plt.show()

    return results

In [73]:
# Esecuzione del benchmark (ridurre `num_trials` se la simulazione quantistica richiede tempi elevati)
benchmark_data = run_benchmark_with_error_tracking(
    alphabet_sizes=[2, 3],
    lengths=[2, 4, 8, 16],
    num_trials=15,
)

[alpha=2, len= 2] Iterazioni medie: 1.47 | Corretti: 15/15 | Errori: 0/15 (0.0%)
[alpha=2, len= 4] Iterazioni medie: 3.33 | Corretti: 15/15 | Errori: 0/15 (0.0%)
[alpha=2, len= 8] Iterazioni medie: 4.13 | Corretti: 9/15 | Errori: 6/15 (40.0%)


Simulation failed and returned the following error message:
ERROR:  [Experiment 0] a circuit requires more memory than max_memory_mb.


AerError: 'Aer simulation failed: ERROR: a circuit requires more memory than max_memory_mb.'